# Sprint 2 - Preference model
Predict a customer's preferred package tier (`ProductPitched`) from their profile. Four models are compared.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src import config
from src.data_loader import load_customers, load_flights, load_attractions, load_hotels
pd.set_option('display.max_columns', 50)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src.train_models import build_preprocessor, get_models

## Data & features

In [ ]:
df = load_customers()
X = df[config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES]
y = df[config.TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE)
print(X_tr.shape, X_te.shape)

## Preprocessing
Numeric: median imputation + scaling. Categorical: most-frequent imputation + one-hot encoding.

In [ ]:
build_preprocessor()

## Train & evaluate all models

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
results, fitted = [], {}
for name, model in get_models().items():
    pipe = Pipeline([('prep', build_preprocessor()), ('model', model)]).fit(X_tr, y_tr)
    p = pipe.predict(X_te)
    results.append({'model': name, 'accuracy': accuracy_score(y_te, p), 'f1_macro': f1_score(y_te, p, average='macro')})
    fitted[name] = pipe
res = pd.DataFrame(results).sort_values('f1_macro', ascending=False).round(4)
res

In [ ]:
res.set_index('model').plot.barh(xlim=(0,1), title='Model comparison'); plt.show()

## Best model

In [ ]:
best = res.iloc[0]['model']
pred = fitted[best].predict(X_te)
print(best)
print(classification_report(y_te, pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_te, pred, labels=[t for t in config.TIER_ORDER if t in set(y)], xticks_rotation=30, cmap='Blues'); plt.show()

## Save the final model
Same script the app uses: cross-validation, reports in `reports/`, best model in `models/`.

In [ ]:
from src.train_models import main
main()

## Observations
- Random Forest is best: accuracy 0.882, macro-F1 0.878 (Gradient Boosting 0.865, Decision Tree 0.839, Logistic Regression 0.700).
- Tree models do much better than Logistic Regression, the tier depends on income/age in a non-linear way.
- MonthlyIncome is by far the most important feature (0.45), then Age (0.15) and NumberOfTrips.
- Most errors are Basic vs Deluxe (64 of 115 test errors), the two biggest classes with similar incomes. King is mostly confused with Super Deluxe and has the lowest recall (0.83, only 230 rows).
